# 01 · Data acquisition
**Project:** LLM-enhanced sequential recommendation on Amazon Beauty (Models 0–4)
**Dataset:** Amazon Reviews 2023, category *Beauty_and_Personal_Care* (McAuley Lab, UCSD)

### What this notebook does
1. Writes the single **`config.json`** that every later notebook reads (seed, k-core, sequence length, thresholds).
2. Downloads the official **5-core ratings** file (`user_id, parent_asin, rating, timestamp`).
3. Downloads the **product metadata** and keeps only products that appear in the 5-core ratings.
4. Validates the schema and row counts against the published statistics.
5. Records **SHA-256 checksums** of the source files so anyone can verify they used identical data.

### Why
The report's Stage 1 requires a *frozen, reproducible* ETL before Model 0 is trained. Reproducibility starts here:
fixed source URLs, checksums and one config file mean every teammate builds exactly the same dataset.

**Why the 5-core file?** It is the official benchmark version (every user and item has ≥ 5 ratings). Users with
fewer purchases cannot form a train / validation / test sequence, so they would be removed anyway.

**Review text is not downloaded here.** The full review file is very large; notebook 06 streams it and keeps only
reviews belonging to the final subset.

**Outputs:** `config.json`, `raw/ratings_5core.parquet`, `raw/meta_5core_items.parquet`, `raw/source_manifest.json`

In [ ]:
# ---------------- Shared setup (identical in every notebook) ----------------
# All notebooks read and write inside one project folder (BASE).
# Colab: BASE lives in your Google Drive so outputs survive runtime restarts.
# Local Jupyter: BASE is ./298A_recsys next to the notebooks.
# Set the environment variable RECSYS_BASE to use any other folder.
import os, sys, json, re, time, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore', category=FutureWarning)

IN_COLAB = 'google.colab' in sys.modules
if os.environ.get('RECSYS_BASE'):
    BASE = Path(os.environ['RECSYS_BASE'])
elif IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/298A_recsys')
else:
    BASE = Path('./298A_recsys').resolve()

RAW, INTERIM, PROCESSED = BASE / 'raw', BASE / 'interim', BASE / 'processed'
REPORTS, FIGS = BASE / 'reports', BASE / 'reports' / 'figures'
# Large downloads go to fast local disk in Colab (not Drive); re-downloaded if the runtime resets.
DOWNLOADS = Path(os.environ.get('RECSYS_DOWNLOAD_DIR', '/content/downloads' if IN_COLAB else str(BASE / 'downloads')))
for d in [RAW, INTERIM, PROCESSED, REPORTS, FIGS, DOWNLOADS]:
    d.mkdir(parents=True, exist_ok=True)

# Plot style: one hue per single-series chart, light recessive grid, no chart junk.
PRIMARY, ACCENT, MUTED = '#2a78d6', '#eb6834', '#8a8984'
plt.rcParams.update({
    'figure.figsize': (8, 4), 'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.edgecolor': '#b5b4ae',
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.labelsize': 9, 'xtick.labelsize': 8,
    'ytick.labelsize': 8, 'axes.prop_cycle': matplotlib.cycler(color=[PRIMARY, ACCENT, '#1baf7a', '#eda100']),
})

def save_json(obj, path):
    Path(path).write_text(json.dumps(obj, indent=2, default=str))

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIGS / f'{name}.png', bbox_inches='tight')
    plt.show()

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

def load_config():
    cfg_path = BASE / 'config.json'
    if not cfg_path.exists():
        raise FileNotFoundError('config.json not found - run 01_data_acquisition.ipynb first.')
    return json.loads(cfg_path.read_text())

print('Project folder:', BASE)

## 1. Project configuration
Every decision that affects the data lives here, so it is written down once and reused by all notebooks.
Change a value **only here**, then re-run notebooks 01 → 07.

In [ ]:
CONFIG = {
    'dataset': 'Amazon Reviews 2023 - Beauty_and_Personal_Care',
    'urls': {
        'ratings_5core': 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/5core/rating_only/Beauty_and_Personal_Care.csv.gz',
        'metadata': 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Beauty_and_Personal_Care.jsonl.gz',
        'reviews': 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Beauty_and_Personal_Care.jsonl.gz',
    },
    # Published 5-core statistics (amazon-reviews-2023.github.io), used as a sanity check.
    'expected_5core': {'users': 729_600, 'items': 207_600, 'ratings': 6_600_000},
    'seed': 42,
    'kcore': 5,                      # min interactions per user and per item
    'max_seq_len': 50,               # standard SASRec length for Amazon Beauty
    'repeat_policy': 'keep_first',   # repeat purchase of the same item by the same user -> keep earliest
    'drop_items_without_title': True,  # all 5 models must share one item set, and Model 1 needs text
    'subset_targets': {'min_users': 20_000, 'max_users': 150_000, 'min_items': 3_000, 'max_items': 50_000},
    'subset_top_categories': 8,      # how many level-1 sub-categories the EDA compares
    'user_sample_sizes': [50_000, 100_000, 150_000],  # random-user subsets the EDA compares
    'n_sampled_negatives': 100,      # secondary (optional) sampled evaluation
    'user_length_bins': [3, 5, 10],  # training-history length bins -> short (3-4), medium (5-9), long (10+)
    'item_pop': {'head_frac': 0.20, 'tail_frac': 0.50},  # top 20% items = head, bottom 50% = tail
    'cold_start': {'item_frac': 0.05, 'newest_pool_frac': 0.30},
    'text': {'max_desc_words': 300, 'max_item_text_words': 256, 'n_review_snippets': 3, 'review_snippet_words': 60},
    'include_review_text': True,
    'eval_ks': [5, 10, 20],
}

# Optional: overrides passed through an environment variable (used for quick test runs).
if os.environ.get('RECSYS_CONFIG_OVERRIDES'):
    CONFIG.update(json.loads(os.environ['RECSYS_CONFIG_OVERRIDES']))

save_json(CONFIG, BASE / 'config.json')
np.random.seed(CONFIG['seed'])
print(json.dumps(CONFIG, indent=2))

## 2. Download helper
Downloads stream to disk in chunks (the files are large), show progress, and are **skipped if already present**.
If the download fails in your environment, download the file manually in a browser and place it at the printed path.

In [ ]:
import requests
try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

def download(url, dest):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        print(f'Already present, skipping download: {dest} ({dest.stat().st_size/1e6:,.1f} MB)')
        return dest
    tmp = dest.with_suffix(dest.suffix + '.part')
    print(f'Downloading {url}\n  -> {dest}')
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        bar = tqdm(total=total, unit='B', unit_scale=True) if tqdm else None
        with open(tmp, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                if bar: bar.update(len(chunk))
        if bar: bar.close()
    tmp.rename(dest)
    return dest

## 3. Ratings (5-core)

In [ ]:
ratings_gz = download(CONFIG['urls']['ratings_5core'], DOWNLOADS / 'Beauty_and_Personal_Care.5core.csv.gz')

ratings = pd.read_csv(ratings_gz, compression='gzip',
                      dtype={'user_id': 'string', 'parent_asin': 'string', 'rating': 'float32', 'timestamp': 'int64'})
expected_cols = ['user_id', 'parent_asin', 'rating', 'timestamp']
assert list(ratings.columns) == expected_cols, f'Unexpected columns: {list(ratings.columns)}'

# row_id = position in the original file; used later as a deterministic tie-breaker for equal timestamps.
ratings['row_id'] = np.arange(len(ratings), dtype=np.int64)
ratings.head()

### Schema and count checks
Counts should be close to the published 5-core statistics. A large gap means a truncated or wrong download.

In [ ]:
n_users, n_items, n_rows = ratings.user_id.nunique(), ratings.parent_asin.nunique(), len(ratings)
exp = CONFIG['expected_5core']
summary = pd.DataFrame({
    'observed': [n_users, n_items, n_rows],
    'published': [exp['users'], exp['items'], exp['ratings']],
}, index=['users', 'items', 'ratings'])
summary['ratio'] = (summary.observed / summary.published).round(3)
display(summary)
if not summary.ratio.between(0.9, 1.1).all():
    print('WARNING: counts differ from the published 5-core statistics by more than 10%. Check the download.')

print('\nDtypes:\n', ratings.dtypes)
print('\nMissing values per column:\n', ratings.isna().sum())
ts_digits = int(np.log10(max(ratings.timestamp.median(), 1))) + 1
print(f'\nTimestamp median has {ts_digits} digits ->', 'milliseconds' if ts_digits >= 13 else 'seconds')

ratings.to_parquet(RAW / 'ratings_5core.parquet', index=False)
print('Saved', RAW / 'ratings_5core.parquet')

## 4. Product metadata
The metadata file covers ~1M products. We stream it line by line and keep only the products present in the
5-core ratings, dropping images and videos (not used by any model). Lists stay as lists; `details` (a dictionary
with varying keys) is stored as a JSON string.

In [ ]:
import gzip

meta_gz = download(CONFIG['urls']['metadata'], DOWNLOADS / 'meta_Beauty_and_Personal_Care.jsonl.gz')
keep_items = set(ratings.parent_asin.unique())
KEEP_FIELDS = ['parent_asin', 'main_category', 'title', 'average_rating', 'rating_number', 'features',
               'description', 'price', 'store', 'categories', 'details']

def as_list(x):
    if x is None: return []
    if isinstance(x, list): return [str(v) for v in x if v is not None]
    return [str(x)]

records, n_lines, n_bad = [], 0, 0
with gzip.open(meta_gz, 'rt', encoding='utf-8') as f:
    for line in f:
        n_lines += 1
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            n_bad += 1
            continue
        if rec.get('parent_asin') not in keep_items:
            continue
        records.append({
            'parent_asin': rec.get('parent_asin'),
            'main_category': rec.get('main_category'),
            'title': rec.get('title'),
            'average_rating': rec.get('average_rating'),
            'rating_number': rec.get('rating_number'),
            'features': as_list(rec.get('features')),
            'description': as_list(rec.get('description')),
            'price': None if rec.get('price') is None else str(rec.get('price')),
            'store': rec.get('store'),
            'categories': as_list(rec.get('categories')),
            'details': json.dumps(rec.get('details') or {}, ensure_ascii=False),
        })
        if n_lines % 200_000 == 0:
            print(f'  scanned {n_lines:,} lines, kept {len(records):,}')

meta = pd.DataFrame(records, columns=KEEP_FIELDS)
meta['average_rating'] = pd.to_numeric(meta['average_rating'], errors='coerce')
meta['rating_number'] = pd.to_numeric(meta['rating_number'], errors='coerce')
print(f'Scanned {n_lines:,} metadata lines ({n_bad} unparsable); kept {len(meta):,} rows for 5-core items')
meta.head(3)

In [ ]:
n_meta_items = meta.parent_asin.nunique()
print(f'5-core items with a metadata record: {n_meta_items:,} / {n_items:,} ({n_meta_items/n_items:.1%})')
print(f'Duplicate parent_asin rows in metadata: {len(meta) - n_meta_items:,} (handled in notebook 03)')
meta.to_parquet(RAW / 'meta_5core_items.parquet', index=False)
print('Saved', RAW / 'meta_5core_items.parquet')

## 5. Source manifest (checksums)
Anyone re-running the pipeline can compare these checksums to confirm they used byte-identical source files.

In [ ]:
source_manifest = {
    'created_utc': pd.Timestamp.now(tz='UTC').isoformat(),
    'files': {
        'ratings_5core': {'url': CONFIG['urls']['ratings_5core'], 'sha256': sha256_file(ratings_gz),
                          'bytes': ratings_gz.stat().st_size, 'rows': int(n_rows)},
        'metadata': {'url': CONFIG['urls']['metadata'], 'sha256': sha256_file(meta_gz),
                     'bytes': meta_gz.stat().st_size, 'lines_scanned': int(n_lines), 'rows_kept': int(len(meta))},
    },
    'versions': {'python': sys.version.split()[0], 'pandas': pd.__version__, 'numpy': np.__version__},
}
save_json(source_manifest, RAW / 'source_manifest.json')
print(json.dumps(source_manifest, indent=2))

## Summary
- `config.json` holds every data decision.
- Raw ratings and the matching metadata are saved as Parquet in `raw/`, with checksums in `raw/source_manifest.json`.

**Next:** `02_eda_and_subset_selection.ipynb`